In [9]:
pip install pandas numpy scipy matplotlib seaborn requests tqdm

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [10]:
import json
import time
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel
from collections import defaultdict
from tqdm.notebook import tqdm
import io
import zipfile

# Set random seed for reproducibility
np.random.seed(42)

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

# Configuration
NUM_DOMAINS = 500          # Number of domains to fetch from Majestic (max 500 for free tier)
NUM_USERS = 20
SEARCHES_PER_USER = 30
NUM_RESULTS = 10

# Click model parameters
RELEVANCE_WEIGHT = 0.7
GREEN_BOOST_WEIGHT = 0.3

# Bandit parameters
ACTIONS = [0, 2, 5]       # Boost levels (0 = no boost, 2 = small, 5 = large)
EPSILON = 0.1
FIXED_BOOST_VALUE = 3     # Fixed boost for the non-adaptive condition

# Caching
GREEN_CACHE_FILE = 'green_cache.json'
DOMAINS_FILE = 'majestic_domains.csv'

/Users/kasmyabhatia/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [11]:
DOMAINS_FILE = "cisco_domains.csv"
NUM_DOMAINS = 1000  # adjust as needed

def fetch_cisco_domains(limit=NUM_DOMAINS):
    """Download top domains from Cisco Umbrella 1 Million."""
    url = "http://s3-us-west-1.amazonaws.com/umbrella-static/top-1m.csv.zip"
    print(f"Downloading Cisco Umbrella dataset from {url} ...")
    response = requests.get(url, stream=True)
    if response.status_code != 200:
        raise Exception(f"Failed to download: {response.status_code}")

    # Read the zip file
    import io, zipfile
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        # The CSV file is named 'top-1m.csv'
        with z.open('top-1m.csv') as f:
            df = pd.read_csv(f, header=None, names=['rank', 'domain'])

    domains = df['domain'].dropna().astype(str).str.lower().tolist()
    # Remove empty and obviously invalid entries
    domains = [d for d in domains if len(d) > 3 and '.' in d]
    return domains[:limit]

# Try to load from cache, otherwise download
try:
    df_domains_cache = pd.read_csv(DOMAINS_FILE)
    domains = df_domains_cache['domain'].tolist()
    print(f"Loaded {len(domains)} domains from cache ({DOMAINS_FILE})")
except FileNotFoundError:
    print("No domain cache found, downloading from Cisco Umbrella...")
    domains = fetch_cisco_domains(limit=NUM_DOMAINS)
    # Save for future runs
    pd.DataFrame({'domain': domains}).to_csv(DOMAINS_FILE, index=False)
    print(f"Saved {len(domains)} domains to {DOMAINS_FILE}")

print(f"Using {len(domains)} domains. First 5: {domains[:5]}")


Loaded 1000 domains from cache (cisco_domains.csv)
Using 1000 domains. First 5: ['google.com', 'gstatic.com', 'www.google.com', 'microsoft.com', 'data.microsoft.com']


In [12]:
def load_green_cache():
    try:
        with open(GREEN_CACHE_FILE, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        return {}

def save_green_cache(cache):
    with open(GREEN_CACHE_FILE, 'w') as f:
        json.dump(cache, f, indent=2)

def is_green(domain, cache):
    if domain in cache:
        return cache[domain]
    
    url = f"https://api.thegreenwebfoundation.org/greencheck/{domain}"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            green = data.get('green', False)
        elif response.status_code == 429:
            print(f"Rate limit hit for {domain}, waiting 5s...")
            time.sleep(5)
            return is_green(domain, cache)  # retry once
        else:
            green = False
    except Exception as e:
        print(f"Error checking {domain}: {e}")
        green = False
    
    cache[domain] = green
    return green

# Load existing cache
green_cache = load_green_cache()

# Check only domains not already cached
print("Fetching green status for domains (cached items will be skipped)...")
for d in tqdm(domains):
    if d not in green_cache:
        is_green(d, green_cache)
        time.sleep(0.2)  # Polite delay to avoid rate limiting

save_green_cache(green_cache)

# Build a DataFrame with domain and green status
domain_green = {d: green_cache[d] for d in domains}
df_domains = pd.DataFrame(list(domain_green.items()), columns=['domain', 'is_green'])
n_green = df_domains['is_green'].sum()
print(f"Green domains: {n_green} out of {len(df_domains)} ({100*n_green/len(df_domains):.1f}%)")
df_domains.head()

Fetching green status for domains (cached items will be skipped)...


  0%|          | 0/1000 [00:00<?, ?it/s]

Green domains: 211 out of 1000 (21.1%)


,domain,is_green
0,google.com,True
1,gstatic.com,True
2,www.google.com,True
3,microsoft.com,False
4,data.microsoft.com,False


In [13]:
class ContextualBandit:
    def __init__(self, epsilon=EPSILON, actions=ACTIONS):
        self.epsilon = epsilon
        self.actions = actions
        # Q[domain][action] = {'value': float, 'count': int}
        self.Q = defaultdict(lambda: {a: {'value': 0.0, 'count': 0} for a in actions})
    
    def select_action(self, domain):
        if np.random.rand() < self.epsilon:
            return np.random.choice(self.actions)
        else:
            domain_q = self.Q[domain]
            # In case of ties, first max is fine
            best_action = max(self.actions, key=lambda a: domain_q[a]['value'])
            return best_action
    
    def update(self, domain, action, reward):
        old = self.Q[domain][action]
        n = old['count'] + 1
        new_value = old['value'] + (reward - old['value']) / n
        self.Q[domain][action] = {'value': new_value, 'count': n}

In [14]:
def generate_search_results(domain_list, green_dict):
    """Randomly sample NUM_RESULTS distinct domains."""
    sampled = np.random.choice(domain_list, size=NUM_RESULTS, replace=False)
    results = []
    for rank, dom in enumerate(sampled, start=1):
        results.append({
            'domain': dom,
            'original_rank': rank,
            'is_green': green_dict[dom]
        })
    return results

def click_probability(result, boost, condition, user_green_pref):
    """
    Compute click probability for a result.
    - relevance: 1 / original_rank
    - green attractiveness: (boost/10) * user_green_pref, but only if green and condition uses boost
    """
    relevance = 1.0 / result['original_rank']
    green_attract = 0.0
    if condition in ['fixed-boost', 'bandit'] and result['is_green']:
        green_attract = (boost / 10.0) * user_green_pref
    prob = RELEVANCE_WEIGHT * relevance + GREEN_BOOST_WEIGHT * green_attract
    # Clamp to avoid extreme values and ensure exploration
    return np.clip(prob, 0.05, 0.95)

def rerank_results(results, condition, bandit=None):
    """Assign boost based on condition and re-sort by original_rank - boost."""
    if condition == 'baseline':
        for r in results:
            r['boost'] = 0
    elif condition == 'fixed-boost':
        for r in results:
            r['boost'] = FIXED_BOOST_VALUE if r['is_green'] else 0
    elif condition == 'bandit' and bandit is not None:
        for r in results:
            r['boost'] = bandit.select_action(r['domain']) if r['is_green'] else 0
    else:
        raise ValueError(f"Unknown condition or missing bandit: {condition}")
    
    # Sort by new score: higher (original_rank - boost) means better position
    # But note: original_rank lower is better, so we sort descending by (boost - original_rank)
    sorted_results = sorted(results, key=lambda x: x['boost'] - x['original_rank'], reverse=True)
    for new_rank, r in enumerate(sorted_results, start=1):
        r['new_rank'] = new_rank
    return sorted_results

In [16]:
def run_session(condition, bandit=None):
    user_green_pref = np.random.uniform(0.5, 1.5)  # user's affinity for green results
    logs = []
    for _ in range(SEARCHES_PER_USER):
        results = generate_search_results(domains, domain_green)
        reranked = rerank_results(results, condition, bandit)
        
        # Log impressions
        for r in reranked:
            logs.append({
                'type': 'impression',
                'domain': r['domain'],
                'original_rank': r['original_rank'],
                'new_rank': r['new_rank'],
                'boost': r['boost'],
                'is_green': r['is_green'],
                'action': r['boost'],
                'condition': condition
            })
        
        # Simulate click (multinomial based on probabilities)
        probs = [click_probability(r, r['boost'], condition, user_green_pref) for r in reranked]
        probs = np.array(probs) / probs.sum()  # normalize to sum 1
        chosen_idx = np.random.choice(len(reranked), p=probs)
        chosen = reranked[chosen_idx]
        
        logs.append({
            'type': 'click',
            'domain': chosen['domain'],
            'new_rank': chosen['new_rank'],
            'was_green': chosen['is_green'],
            'condition': condition
        })
        
        # Update bandit (only if we clicked a green domain that received a positive boost)
        if condition == 'bandit' and bandit and chosen['is_green'] and chosen['boost'] > 0:
            # Reward = 1 for a click on a green result that was boosted
            bandit.update(chosen['domain'], chosen['boost'], reward=1)
    
    return logs